# A first-class `pymc-extras` statespace forecaster

This example fits the same noisy local-level model in two ways:

1. `markov_time_series` represents every level as a scan-backed latent sampled by NUTS.
2. `pymc_extras.statespace` marginalizes those levels with a Kalman filter and samples only the model parameters.

`StatespaceForecaster` maps the second lifecycle onto the same `fit`, `forecast`, `backtest`, and metric APIs as the hand-written model. The comparison below records parameter recovery, forecast quality, and wall time on the same synthetic data.

In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import xarray as xr
from pymc_extras.statespace.models import structural as st

from pymc_forecast import (
    HMCForecaster,
    StatespaceForecaster,
    backtest,
    evaluate_forecast,
    markov_time_series,
    predict,
)

## Synthetic local-level data

The latent level follows a Gaussian random walk and observations add independent Gaussian noise. The last 12 days are held out.

In [ ]:
SEED = 20260713
N_TRAIN = 80
HORIZON = 12
SIGMA_LEVEL = 0.12
SIGMA_OBS = 0.08

rng = np.random.default_rng(SEED)
dates = pd.date_range("2025-01-01", periods=N_TRAIN + HORIZON, freq="D")
level = 5.0 + np.cumsum(rng.normal(0.0, SIGMA_LEVEL, len(dates)))
observed = level + rng.normal(0.0, SIGMA_OBS, len(dates))
series = xr.DataArray(observed, dims="time", coords={"time": dates}, name="y")
train = series.isel(time=slice(None, N_TRAIN))
test = series.isel(time=slice(N_TRAIN, None)).rename({"time": "time_future"})

series.to_pandas().plot(figsize=(10, 3), title="Noisy local-level series")

## Scan-based model

The transition distribution is explicit, so NUTS samples the complete in-sample `level` trajectory. The separate `level_future` trajectory is seeded from its final posterior level during forecasting.

In [ ]:
def scan_local_level(h, covariates):
    initial_level = pm.Normal("initial_level", 5.0, 1.0)
    sigma_level = pm.HalfNormal("sigma_level", 0.5)
    sigma_obs = pm.HalfNormal("sigma_obs", 0.5)
    level = markov_time_series(
        h,
        "level",
        init=initial_level,
        transition=lambda previous, sigma: pm.Normal.dist(previous, sigma),
        params=(sigma_level,),
    )
    predict(
        h,
        lambda name, mu, dims, observed: pm.Normal(
            name, mu, sigma_obs, dims=dims, observed=observed
        ),
        level,
    )

## Marginalized statespace model

A statespace builder receives the labeled training slice and returns a fresh statespace/PyMC-model pair. Freshness matters in rolling backtests because `build_statespace_graph` stores the data and coordinates used by later forecasts.

In [ ]:
def statespace_local_level(data, covariates):
    ss_model = (st.LevelTrend(order=1, innovations_order=1) + st.MeasurementError()).build(
        verbose=False
    )

    with pm.Model(coords=ss_model.coords) as model:
        pm.Normal(
            "initial_level_trend",
            mu=float(data.isel(time=0)),
            sigma=1.0,
            dims=ss_model.param_dims["initial_level_trend"],
        )
        pm.HalfNormal(
            "sigma_level_trend",
            sigma=0.5,
            dims=ss_model.param_dims["sigma_level_trend"],
        )
        pm.HalfNormal("sigma_MeasurementError", sigma=0.5)
        pm.Deterministic("P0", pt.eye(ss_model.k_states), dims=ss_model.param_dims["P0"])
        ss_model.build_statespace_graph(data.to_pandas(), mvn_method="cholesky")

    return ss_model, model

## Fit and forecast

Both fits use the same NUTS schedule. These settings keep the example quick; use longer chains for final inference. The statespace fit has only three sampled parameters, whereas the scan fit also samples 80 correlated levels.

In [ ]:
fit_options = {
    "draws": 200,
    "tune": 300,
    "chains": 2,
    "random_seed": SEED,
}

start = perf_counter()
scan_forecaster = HMCForecaster(scan_local_level, train, **fit_options)
scan_fit_seconds = perf_counter() - start

start = perf_counter()
statespace_forecaster = StatespaceForecaster(
    statespace_local_level,
    train,
    forecast_kwargs={"mvn_method": "cholesky"},
    **fit_options,
)
statespace_fit_seconds = perf_counter() - start

scan_prediction = scan_forecaster.forecast(horizon=HORIZON, num_samples=400, random_seed=SEED)[
    "predictions"
]["forecast"]
statespace_prediction = statespace_forecaster.forecast(
    horizon=HORIZON, num_samples=400, random_seed=SEED
)["predictions"]["forecast"]

## Posterior quality and runtime

Both parameterizations recover the two innovation scales. Exact values vary with the generated series and short chains; the point of the comparison is that Kalman marginalization removes the strongly correlated per-time latent vector from NUTS.

In [ ]:
scan_posterior = scan_forecaster.idata["posterior"]
statespace_posterior = statespace_forecaster.idata["posterior"]
comparison = pd.DataFrame(
    {
        "truth": [SIGMA_LEVEL, SIGMA_OBS],
        "scan posterior mean": [
            float(scan_posterior["sigma_level"].mean()),
            float(scan_posterior["sigma_obs"].mean()),
        ],
        "statespace posterior mean": [
            float(statespace_posterior["sigma_level_trend"].mean()),
            float(statespace_posterior["sigma_MeasurementError"].mean()),
        ],
    },
    index=["level innovation", "observation noise"],
)
runtime = pd.Series(
    {"scan seconds": scan_fit_seconds, "statespace seconds": statespace_fit_seconds}
)
comparison, runtime.round(2)

In [ ]:
scores = pd.DataFrame(
    {
        "scan": evaluate_forecast(scan_prediction, test),
        "statespace": evaluate_forecast(statespace_prediction, test),
    }
)
scores

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
train.to_pandas().plot(ax=ax, color="black", label="train")
test.rename({"time_future": "time"}).to_pandas().plot(
    ax=ax, color="black", linestyle="--", label="held out"
)
for name, prediction, color in [
    ("scan", scan_prediction, "C0"),
    ("statespace", statespace_prediction, "C1"),
]:
    mean = prediction.mean(("chain", "draw"))
    interval = prediction.quantile([0.05, 0.95], dim=("chain", "draw"))
    ax.plot(mean["time_future"], mean, color=color, label=name)
    ax.fill_between(
        mean["time_future"],
        interval.sel(quantile=0.05),
        interval.sel(quantile=0.95),
        color=color,
        alpha=0.2,
    )
ax.legend()
ax.set_title("Local-level forecasts: sampled scan vs marginalized statespace")

## The same backtest and metrics API

The adapter is selected with `forecaster_cls`; window construction, timing, prediction retention, and metrics are unchanged. The builder creates a fresh statespace object for every window.

In [ ]:
statespace_backtest = backtest(
    series,
    None,
    statespace_local_level,
    forecaster_cls=StatespaceForecaster,
    min_train_window=80,
    test_window=6,
    stride=6,
    num_samples=100,
    forecaster_options={
        "draws": 100,
        "tune": 150,
        "chains": 1,
        "forecast_kwargs": {"mvn_method": "cholesky"},
    },
    keep_predictions=True,
    random_seed=SEED,
)
pd.DataFrame(
    [
        {"split": result.t1, **result.metrics, "fit seconds": result.train_walltime}
        for result in statespace_backtest
    ]
)

## Takeaway

Use `markov_time_series` when the transition is nonlinear, non-Gaussian, or otherwise outside the Kalman-filter family. For linear-Gaussian structural models, `StatespaceForecaster` keeps the same package-level workflow while letting `pymc-extras` marginalize the time-indexed states.